# Glove vs Fasttext vs Word2Vec

In [ ]:
import numpy as np
import pandas as pd
import gensim.downloader as api

from pathlib import Path
from collections import Counter
from nltk.tokenize import TreebankWordTokenizer
from src.utils import load_datasets, PretrainedEmbeddings

In [4]:
train_df, val_df, test_df = load_datasets()

In [5]:
def download_pretrained(option):
    """
    You can use this to download GloVe (or similar) model using wget.
    It's fine to download it through other means, but don't unzip the file (the rest of the notebook can
    handle the file in its zipped format).

    This function returns the path which you can use for the loader class that follows.

    The function will skip the download if it detects the file in disk. If the file in disk is corrupted, you will have to delete the file first.
    """
    if option == "glove_50":
        destination = "glove.6B.50d.txt"
        dims = 50
        skip_first = False

    elif option == "fasttext":
        destination = "wiki-news-300d-1M.vec.zip"
        dims = 300
        skip_first = True

    else:
        raise ValueError(f"Unknown option: {option}")


    if not Path(destination).is_file():

        if option == "glove_50":
            !wget -O glove.6B.50d.txt https://huggingface.co/kcz358/glove/resolve/main/glove.6B/glove.6B.50d.txt

        elif option == "fasttext":
            !wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip

        if not Path(destination).is_file():
            raise FileNotFoundError(f"Download failed: {destination}")

    return destination, dims, skip_first

**Embedding model proposals**

In [6]:
# Compare the vocab size of the models

models = {}

for option in ["glove_50", "fasttext", "word2vec"]:

    if option != "word2vec":
        path, dims, skip_first = download_pretrained(option)

        models[option] = PretrainedEmbeddings(
            path, dims, skip_first_line=skip_first
        )

    else:
        models[option] = api.load("word2vec-google-news-300")

    if option == "word2vec":
        dims = models[option].vector_size
        vocab_size = len(models[option].key_to_index)
    else:
        dims = models[option].num_dims
        vocab_size = len(models[option].vocab)

    print(option, "dimensions:", dims, "vocabulary size:", vocab_size)

glove_50 dimensions: 50 vocabulary size: 400002
fasttext dimensions: 300 vocabulary size: 830463
word2vec dimensions: 300 vocabulary size: 3000000


Word Tokenization for the dataset

In [7]:
tokenizer = TreebankWordTokenizer()

def tokenize(text):
    tokens = tokenizer.tokenize(text)

    return [
        token for token in tokens
        if any(char.isalnum() for char in token) # removes punctuation (because of word2vec mainly)
    ]

# Example tokenization
for sentence in train_df["sentence1"].head(5):
    print("Original:", sentence)
    print("Tokens:", tokenize(sentence), "\n")

Original: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Tokens: ['In', 'Paris', 'in', 'October', '1560', 'he', 'secretly', 'met', 'the', 'English', 'ambassador', 'Nicolas', 'Throckmorton', 'asking', 'him', 'for', 'a', 'passport', 'to', 'return', 'to', 'England', 'through', 'Scotland'] 

Original: The NBA season of 1975 -- 76 was the 30th season of the National Basketball Association .
Tokens: ['The', 'NBA', 'season', 'of', '1975', '76', 'was', 'the', '30th', 'season', 'of', 'the', 'National', 'Basketball', 'Association'] 

Original: There are also specific discussions , public profile debates and project discussions .
Tokens: ['There', 'are', 'also', 'specific', 'discussions', 'public', 'profile', 'debates', 'and', 'project', 'discussions'] 

Original: When comparable rates of flow can be maintained , the results are high .
Tokens: ['When', 'comparable', 'rates', 'of', 'flow

In [8]:
sentences = pd.concat([
    train_df["sentence1"],
    train_df["sentence2"]
], ignore_index=True)

tokens = [
    token
    for sentence in sentences
    for token in tokenize(sentence)
]

print("Total tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))

Total tokens: 1824862
Unique tokens: 35294


**Compare vocabulary coverage for the embedding models**

In [9]:
def vocabulary_coverage(model, tokens):
    unique_tokens = set(tokens)

    covered_total_tokens = sum(token in model for token in tokens)
    covered_unique_tokens = sum(token in model for token in unique_tokens)

    total_token_coverage = covered_total_tokens / len(tokens)
    unique_token_coverage = covered_unique_tokens / len(unique_tokens)

    oov_words = [
        word for word in unique_tokens
        if word not in model
    ]

    return {
        "total_token_coverage": total_token_coverage,
        "unique_token_coverage": unique_token_coverage,
        "oov_words": oov_words
    }

In [10]:
results = {}

for name, model in models.items():
    results[name] = vocabulary_coverage(model, tokens)

coverage_df = pd.DataFrame({
    name: {
        "Total token coverage": result["total_token_coverage"] * 100,
        "Unique token coverage": result["unique_token_coverage"] * 100
    }
    for name, result in results.items()
}).T

display(coverage_df)

,Total token coverage,Unique token coverage
glove_50,97.913760,87.312291
fasttext,98.602305,91.134470
word2vec,83.471024,81.962940


**Missing token examples**

In [11]:
token_counts = Counter(tokens)

for name, result in results.items():
    print(f"{name}: Most frequent missing tokens")

    missing = sorted(
        result["oov_words"],
        key=lambda word: token_counts[word],
        reverse=True
    )

    for word in missing[:8]:
        print(word, token_counts[word])
    print()

glove_50: Most frequent missing tokens
Berasia 136
Urechioiu 48
Eoacmaea 48
Aganbegyan 48
Eletrônica 48
Garré 48
Shantih 48
Lopham 48

fasttext: Most frequent missing tokens
Lottiidae 80
Urechioiu 48
Eoacmaea 48
Aganbegyan 48
Eletrônica 48
UPFA-SLFP 48
Eoacmaeidae 48
Leudesius 48

word2vec: Most frequent missing tokens
and 65840
of 60405
a 35172
to 27754
's 5338
2011 1201
2015 1082
2010 954



# Question 2

Cosine similarity of different word types

In [ ]:
def get_vector(word, model, name):
    if name == "word2vec":
        if word in model:
            return model[word]
        return None

    if word in model:
        return model.embed_word(word)
    return None


def cos_similarity(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))


def word_similarity(word1, word2, model, name):
    u = get_vector(word1, model, name)
    v = get_vector(word2, model, name)

    if u is None or v is None:
        return np.nan

    return cos_similarity(u, v)

In [17]:
word_pairs = [
    ("car", "automobile", "Synonyms"),
    ("city", "town", "Synonyms"),
    ("car", "banana", "Unrelated"),
    ("city", "elephant", "Unrelated"),
    ("dog", "cat", "Related concepts"),
    ("king", "queen", "Related concepts"),
    ("increase", "decrease", "Antonyms"),
    ("before", "after", "Antonyms"),
]

rows = []

for word1, word2, category in word_pairs:
    row = {
        "Word 1": word1,
        "Word 2": word2,
        "Category": category
    }

    for name, model in models.items():
        row[name] = word_similarity(word1, word2, model, name)

    rows.append(row)

similarity_df = pd.DataFrame(rows)
display(similarity_df.round(3))

,Word 1,Word 2,Category,glove_50,fasttext,word2vec
0,car,automobile,Synonyms,0.696,0.767,0.584
1,city,town,Synonyms,0.869,0.799,0.672
2,car,banana,Unrelated,0.219,0.391,0.112
3,city,elephant,Unrelated,0.272,0.384,0.104
4,dog,cat,Related concepts,0.922,0.687,0.761
5,king,queen,Related concepts,0.784,0.667,0.651
6,increase,decrease,Antonyms,0.892,0.923,0.837
7,before,after,Antonyms,0.951,0.784,0.758


Closest words to a certain word

In [ ]:
def nearest_neighbors(word, model, name, k=10):
    if name == "word2vec":
        if word not in model:
            word = word.lower()

        if word not in model:
            return []

        return model.most_similar(word, topn=k)

    if word not in model:
        return []

    query = model.embed_word(word)

    E = model.vectors

    query_norm = np.linalg.norm(query)
    norms = np.linalg.norm(E, axis=1)

    scores = (E @ query) / (np.maximum(norms * query_norm, 1e-12))

    # Exclude special tokens and the query word
    scores[:2] = -np.inf
    scores[model.index(word)] = -np.inf

    top_indices = np.argsort(scores)[-k:][::-1]

    return [(model.word(i), float(scores[i])) for i in top_indices]

test_words = ["car", "city", "king", "increase"]

for word in test_words:
    print(f"\nWORD: {word}")

    for name, model in models.items():
        neighbors = nearest_neighbors(word, model, name, k=5)

        print(f"{name}: {[w for w, score in neighbors]}")


WORD: car
glove_50: ['truck', 'cars', 'vehicle', 'driver', 'driving']
fasttext: ['cars', 'automobile', 'vehicle', 'truck', 'suv']
word2vec: ['vehicle', 'cars', 'SUV', 'minivan', 'truck']

WORD: city
glove_50: ['town', 'downtown', 'where', 'cities', 'area']
fasttext: ['town', 'cities', 'downtown', 'towns', 'suburb']
word2vec: ['citys', 'mayor', 'town', 'municipality', 'municipal']

WORD: king
glove_50: ['prince', 'queen', 'ii', 'emperor', 'son']
fasttext: ['kings', 'queen', 'alveda', 'burger', 'king-']
word2vec: ['kings', 'queen', 'monarch', 'crown_prince', 'prince']

WORD: increase
glove_50: ['increases', 'increased', 'increasing', 'decrease', 'reduction']
fasttext: ['decrease', 'reduce', 'increases', 'increased', 'increasing']
word2vec: ['decrease', 'increases', 'increased', 'reduction', 'increasing']


Average embedding (shows formally that word vectors dont encode positions/context)

In [ ]:
def average_embedding(sentence, model, name):
    vectors = [get_vector(token, model, name) for token in tokenize(sentence)]

    if not vectors:
        return None

    return np.mean(vectors, axis=0)


sentence1 = "The dog chased the man."
sentence2 = "The man chased the dog."

for name, model in models.items():
    v1 = average_embedding(sentence1, model, name)
    v2 = average_embedding(sentence2, model, name)

    print(name, cos_similarity(v1, v2))

glove_50 0.99999994
fasttext 1.0000001
word2vec 0.99999994


Overall limitations/findings:
- Word vectors can't encode the context of the sentence and position
- Antonym vectors are similar for all models (could be problematic for PAWS specifically)
- Word2Vec model struggles with stopwords and numbers even after filtering for punctuation

I think Glove would probably be the best so far, as we can also motivate that it performs similarly to fasttext but is a smaller model